# US Domestic Fare Estimation — With an Honest Look at Forecasting

**CS 451 — End-to-End Data Science Final Project**

This notebook builds a complete pipeline for **estimating** US domestic airfares from the DOT Consumer Airfare Report, and includes an honest failure-case analysis showing where the model *cannot* be used as a forecaster.

**Data Source:** [DOT Consumer Airfare Report: Table 1 — Top 1,000 Contiguous State City-Pair Markets](https://data.transportation.gov/Aviation/Consumer-Airfare-Report-Table-1-Top-1-000-Contiguo/4f3n-jbg2)

---

### What this model does (and doesn't) do

**Does:** Given a route (city1 → city2), carrier, year, and quarter that the model has seen examples of during training, estimate the average fare. This is a **cross-sectional estimation** problem — filling in cells of a route • carrier • time grid from historical patterns.

**Does not:** Forecast fares into the future. Tree-based models cannot extrapolate beyond their training range — when asked about 2026, a boosted-tree model will predict something close to its 2025 behavior regardless of underlying trend. We demonstrate this failure mode explicitly in §10.5.

---

### Pipeline
1. Data acquisition from DOT Socrata API
2. Data quality audit with quantified before/after cleaning
3. EDA — 7 visualizations with interpretation
4. 13 engineered features (8 DOT-derived + 5 external passthrough, no target leakage)
5. 60/20/20 train/val/test split
6. Three model families: **Ridge (linear)**, **Random Forest (bagging)**, **HistGradientBoosting (boosting)**
7. 5-fold CV + `RandomizedSearchCV` hyperparameter tuning
8. Permutation importance for feature selection
9. Final held-out test evaluation
10. **Temporal holdout analysis** — can the model forecast? (Spoiler: no, and we show why)
11. Gradio deployment as a **fare estimator** (year capped at training range)

**Reproducibility:** `RANDOM_STATE = 42` everywhere. Data pulled live from the DOT API — no local files needed.
**Ethics:** Public-domain US government data, no PII, aggregated market statistics only.


## 0. Imports & Config

In [ ]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Utilities
import pickle
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100


## 1. Data Acquisition

Pulled live from the DOT Socrata API. Each row is one (city1, city2, year, quarter) observation with the average fare, the largest carrier's fare + market share, the lowest-fare carrier's fare + market share, route distance, and passenger count.

In [ ]:
# Pull full dataset from DOT Socrata API
url = "https://data.transportation.gov/api/views/4f3n-jbg2/rows.csv?accessType=DOWNLOAD"
df_raw = pd.read_csv(url)

print(f"Raw shape: {df_raw.shape}")
print(f"Columns ({len(df_raw.columns)}):")
for i, col in enumerate(df_raw.columns):
    print(f"  {i+1:2d}. {col:20s}  ({df_raw[col].dtype})")


### 1.1 Sample rows & summary statistics

In [ ]:
print("=== FIRST 3 ROWS ===")
print(df_raw[['Year', 'quarter', 'city1', 'city2', 'nsmiles',
              'passengers', 'fare', 'carrier_lg', 'large_ms', 'fare_lg',
              'carrier_low', 'lf_ms', 'fare_low']].head(3).to_string())

print("\n=== NUMERIC SUMMARY ===")
print(df_raw[['fare', 'fare_lg', 'fare_low', 'nsmiles', 'passengers',
              'large_ms', 'lf_ms']].describe().round(2))


### 1.2 Data Quality Audit
Quantify every data quality issue before deciding how to handle each one.

In [ ]:
print("=== SHAPE & COVERAGE ===")
print(f"Rows: {len(df_raw):,}")
print(f"Year range: {df_raw['Year'].min()}–{df_raw['Year'].max()}")
print(f"Quarters: {sorted(df_raw['quarter'].unique())}")
print(f"Unique city pairs: {df_raw[['city1','city2']].drop_duplicates().shape[0]}")
print(f"Unique carriers (lg): {df_raw['carrier_lg'].nunique()}")
print(f"Unique carriers (low): {df_raw['carrier_low'].nunique()}")

print("\n=== MISSING VALUES ===")
miss = df_raw.isnull().sum()
print(miss[miss > 0] if miss.sum() else "No missing values")

print(f"\n=== DUPLICATES ===")
print(f"Duplicate rows: {df_raw.duplicated().sum()}")

print(f"\n=== OUTLIER CHECK (fare) ===")
q1, q3 = df_raw['fare'].quantile([0.25, 0.75])
iqr = q3 - q1
low_bound, high_bound = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = ((df_raw['fare'] < low_bound) | (df_raw['fare'] > high_bound)).sum()
print(f"IQR bounds: [{low_bound:.2f}, {high_bound:.2f}]")
print(f"Fare outliers (IQR rule): {outliers:,} ({outliers/len(df_raw)*100:.2f}%)")
print(f"Fare min/max: ${df_raw['fare'].min():.2f} / ${df_raw['fare'].max():.2f}")
print(f"Distance min/max: {df_raw['nsmiles'].min()} / {df_raw['nsmiles'].max()} mi")


## 2. Data Cleaning

**Decisions & justifications:**

1. **Drop metadata columns** — `Geocoded_City1/2`, `tbl1apk`, `citymarketid_1/2`, `table_1_flag` are record-keeping fields, not predictive signal.
2. **Drop leakage columns** — `fare_lg`, `fare_low`, `lf_ms` are the outputs we're predicting, disguised. Keeping them would cause severe data leakage. `carrier_low` is also leaky (determined ex-post by observed fares), so we drop it and rely only on `carrier_lg` and `large_ms` which are pre-period market structure signals.
3. **Filter to 2010+** — fares span 1996–2025 in nominal dollars. A $200 fare in 1996 is not a $200 fare in 2025. Rather than engineer a CPI deflator (error-prone with hardcoded tables), we restrict to the modern low-cost-carrier era (2010+). This still leaves ~40K rows, well above the 1,000-sample floor.
4. **Drop NaN rows** — very few (< 0.01%), so deletion is cheaper than imputation.
5. **Clip fare outliers** — cap at 99th percentile. Keeping extreme outliers (often data entry errors on low-volume routes) would dominate MAE/RMSE and destabilize tree splits. Capping preserves sample size while limiting their influence.

In [ ]:
# ---- BEFORE stats ----
before = {
    'rows': len(df_raw),
    'columns': df_raw.shape[1],
    'fare_mean': df_raw['fare'].mean(),
    'fare_std': df_raw['fare'].std(),
    'fare_max': df_raw['fare'].max(),
    'missing_cells': df_raw.isnull().sum().sum(),
    'duplicate_rows': df_raw.duplicated().sum(),
}

# ---- Cleaning ----
df = df_raw.copy()

# 1. Drop metadata / id columns (keep only features + target)
meta_cols = ['Geocoded_City1', 'Geocoded_City2', 'tbl1apk', 'table_1_flag',
             'citymarketid_1', 'citymarketid_2']
df = df.drop(columns=[c for c in meta_cols if c in df.columns])

# 2. Drop leakage columns (known only because we observe the target)
leak_cols = ['fare_lg', 'fare_low', 'lf_ms', 'carrier_low']
df = df.drop(columns=[c for c in leak_cols if c in df.columns])

# 3. Filter to modern era (reduce inflation confound)
df = df[df['Year'] >= 2010].copy()

# 4. Drop any remaining NaN
df = df.dropna()

# 5. Drop duplicates
df = df.drop_duplicates()

# 6. Cap fare outliers at 99th percentile
fare_99 = df['fare'].quantile(0.99)
n_capped = (df['fare'] > fare_99).sum()
df['fare'] = df['fare'].clip(upper=fare_99)

# ---- AFTER stats ----
after = {
    'rows': len(df),
    'columns': df.shape[1],
    'fare_mean': df['fare'].mean(),
    'fare_std': df['fare'].std(),
    'fare_max': df['fare'].max(),
    'missing_cells': df.isnull().sum().sum(),
    'duplicate_rows': df.duplicated().sum(),
}

# ---- Before/After comparison ----
comparison = pd.DataFrame([before, after], index=['Before', 'After']).T
comparison['Change'] = comparison['After'] - comparison['Before']
print("=== BEFORE vs AFTER CLEANING ===")
print(comparison.round(2).to_string())
print(f"\nFares capped at 99th percentile (${fare_99:.2f}): {n_capped:,} rows affected")
print(f"\nFinal feature columns retained: {df.columns.tolist()}")


## 2.5 External Data Augmentation

Now that the DOT primary dataset is cleaned (duplicates removed, NaN dropped, outliers capped, filtered to 2010+), we enrich every row with two free public sources. Doing the augmentation **after** cleaning means we only fetch external data for the rows that will actually be used downstream.

| Source | What | Join key | Why it matters |
|---|---|---|---|
| **EIA jet fuel spot price** | $/gallon, Gulf Coast FOB | Year + quarter | Fuel is ~25% of airline costs |
| **BLS CPI (All Urban)** | Monthly inflation index | Year + quarter | Captures general inflation pressure |

**Architecture:** DOT remains the primary dataset — every row in our training data is a DOT row. The external sources add new *columns* (not new rows) through lookups on the join keys. Target variable, observation unit, and sample size are unchanged.

**Caching strategy:** All external downloads are cached to Google Drive on first run. Subsequent reruns read from cache in seconds rather than re-hitting APIs.

**Scope decision — supply-side data.** We considered adding BTS T-100 Segment data (load factor, capacity per route per carrier) but deferred it to future work. BTS's bulk download endpoint requires a session-based web form flow (dynamic URLs generated per-request with ASP.NET session cookies) that didn't fit our time budget. Load factor would have added supply-side visibility; we discuss this as a natural extension in the Future Work section.

### 2.5.1 Setup — API keys and cache paths

In [ ]:
# Colab-specific: get EIA API key from userdata (Secrets sidebar)
# In the left sidebar, click the key icon, add EIA_API_KEY there.
from google.colab import userdata, drive
import os

try:
    EIA_API_KEY = userdata.get('EIA_API_KEY')
    print("EIA_API_KEY loaded from Colab secrets.")
except Exception:
    raise RuntimeError(
        "EIA_API_KEY not found. Go to the Colab left sidebar, click the key icon, "
        "add a secret named EIA_API_KEY with your API key, and toggle notebook "
        "access on. Free key: https://www.eia.gov/opendata/")

# BLS API works without a key at low volume; register for higher limits if needed.
try:
    BLS_API_KEY = userdata.get('BLS_API_KEY')
    print("BLS_API_KEY loaded.")
except Exception:
    BLS_API_KEY = None
    print("BLS_API_KEY not set — using unregistered access (low rate limit, fine for this use).")

# Cache directory on Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

CACHE_DIR = '/content/drive/MyDrive/Colab Notebooks/Final project/external_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Cache directory: {CACHE_DIR}")


### 2.5.2 EIA Jet Fuel Spot Prices

**Series:** `PET.EER_EPJK_PF4_RGC_DPG.W` (U.S. Gulf Coast Kerosene-Type Jet Fuel Spot Price FOB, weekly).

Weekly spot prices in $/gallon from 1990 to present. We aggregate to quarterly means to match the DOT granularity. We also engineer a 1-quarter lag feature to capture the delayed pass-through from fuel costs to fares (airlines hedge and capacity/pricing decisions take time to adjust).

In [ ]:
import requests
import pandas as pd
import numpy as np

EIA_CACHE = os.path.join(CACHE_DIR, 'eia_jet_fuel.parquet')

def fetch_eia_jet_fuel():
    """Fetch weekly jet fuel spot price from EIA API v2."""
    url = "https://api.eia.gov/v2/petroleum/pri/spt/data/"
    params = {
        'api_key': EIA_API_KEY,
        'frequency': 'weekly',
        'data[0]': 'value',
        'facets[series][]': 'EER_EPJK_PF4_RGC_DPG',
        'start': '2010-01-01',
        'sort[0][column]': 'period',
        'sort[0][direction]': 'asc',
        'length': 5000,
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    records = r.json()['response']['data']
    fuel = pd.DataFrame(records)[['period', 'value']]
    fuel.columns = ['date', 'jet_fuel_usd_per_gal']
    fuel['date'] = pd.to_datetime(fuel['date'])
    fuel['jet_fuel_usd_per_gal'] = pd.to_numeric(fuel['jet_fuel_usd_per_gal'])
    return fuel

# Use cache if present
if os.path.exists(EIA_CACHE):
    fuel_weekly = pd.read_parquet(EIA_CACHE)
    print(f"Loaded EIA fuel data from cache: {len(fuel_weekly)} weekly observations")
else:
    print("Fetching EIA jet fuel data from API...")
    fuel_weekly = fetch_eia_jet_fuel()
    fuel_weekly.to_parquet(EIA_CACHE)
    print(f"Fetched and cached {len(fuel_weekly)} weekly observations")

print(f"Date range: {fuel_weekly['date'].min().date()} to {fuel_weekly['date'].max().date()}")
print(f"Price range: ${fuel_weekly['jet_fuel_usd_per_gal'].min():.2f} to "
      f"${fuel_weekly['jet_fuel_usd_per_gal'].max():.2f} per gallon")
print(fuel_weekly.head())


In [ ]:
# Aggregate to quarterly and engineer lag feature
fuel_weekly['Year'] = fuel_weekly['date'].dt.year
fuel_weekly['quarter'] = fuel_weekly['date'].dt.quarter

fuel_quarterly = (fuel_weekly
                  .groupby(['Year', 'quarter'])['jet_fuel_usd_per_gal']
                  .mean()
                  .reset_index())

# 1-quarter lag (captures delayed pass-through)
fuel_quarterly = fuel_quarterly.sort_values(['Year', 'quarter']).reset_index(drop=True)
fuel_quarterly['jet_fuel_lag_1q'] = fuel_quarterly['jet_fuel_usd_per_gal'].shift(1)

# YoY change (captures fuel price shocks)
fuel_quarterly['jet_fuel_yoy_change'] = (
    fuel_quarterly['jet_fuel_usd_per_gal'].pct_change(periods=4) * 100
)

print(f"Quarterly fuel data: {len(fuel_quarterly)} rows")
print(fuel_quarterly.tail(8).to_string(index=False))

# Quick viz: fuel price over time
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 3.5))
fuel_quarterly['period'] = fuel_quarterly['Year'] + (fuel_quarterly['quarter'] - 1) / 4
ax.plot(fuel_quarterly['period'], fuel_quarterly['jet_fuel_usd_per_gal'],
        color='#D62728', linewidth=1.8)
ax.set_title('U.S. Gulf Coast Jet Fuel Spot Price (Quarterly Mean)')
ax.set_xlabel('Year')
ax.set_ylabel('USD per Gallon')
ax.grid(True, alpha=0.3)
ax.axvspan(2020, 2021.5, alpha=0.15, color='gray', label='COVID collapse')
ax.legend()
plt.tight_layout()
plt.show()


### 2.5.3 BLS Consumer Price Index

**Series:** `CUUR0000SA0` (CPI-U, All Items, U.S. City Average, Not Seasonally Adjusted).

The standard inflation measure. Published monthly on or around the 10th of the following month. We aggregate to quarterly means and derive a year-over-year inflation rate feature.

Per the planning discussion, we keep nominal fares as the target and let the model learn the inflation relationship itself by exposing CPI as a feature — interesting empirical question whether the model picks it up.

In [ ]:
BLS_CACHE = os.path.join(CACHE_DIR, 'bls_cpi.parquet')

def fetch_bls_cpi(start_year=2009, end_year=2025):
    """BLS Public Data API v2. Batches of 20-year windows allowed per call;
    our 2009-2025 window is one call. Key is optional at low volume."""
    url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"
    payload = {
        'seriesid': ['CUUR0000SA0'],
        'startyear': str(start_year),
        'endyear': str(end_year),
    }
    if BLS_API_KEY:
        payload['registrationkey'] = BLS_API_KEY
    r = requests.post(url, json=payload, timeout=30)
    r.raise_for_status()
    series = r.json()['Results']['series'][0]['data']
    rows = []
    for obs in series:
        # period like 'M01'..'M12'
        if obs['period'].startswith('M'):
            rows.append({
                'Year':  int(obs['year']),
                'month': int(obs['period'][1:]),
                'cpi':   float(obs['value']),
            })
    return pd.DataFrame(rows).sort_values(['Year', 'month']).reset_index(drop=True)

if os.path.exists(BLS_CACHE):
    cpi_monthly = pd.read_parquet(BLS_CACHE)
    print(f"Loaded BLS CPI from cache: {len(cpi_monthly)} monthly observations")
else:
    print("Fetching BLS CPI from API...")
    cpi_monthly = fetch_bls_cpi(2009, 2025)
    cpi_monthly.to_parquet(BLS_CACHE)
    print(f"Fetched and cached {len(cpi_monthly)} monthly observations")

print(f"CPI range: {cpi_monthly['cpi'].min():.1f} to {cpi_monthly['cpi'].max():.1f}")
print(f"Interpretation: CPI of 313 means overall prices are 3.13× the 1982-84 average.")
print(cpi_monthly.tail(6).to_string(index=False))


In [ ]:
# Aggregate to quarterly + derive YoY inflation
cpi_monthly['quarter'] = ((cpi_monthly['month'] - 1) // 3) + 1
cpi_quarterly = (cpi_monthly
                 .groupby(['Year', 'quarter'])['cpi']
                 .mean()
                 .reset_index())

cpi_quarterly = cpi_quarterly.sort_values(['Year', 'quarter']).reset_index(drop=True)
cpi_quarterly['cpi_yoy_change'] = cpi_quarterly['cpi'].pct_change(periods=4) * 100

print(f"Quarterly CPI: {len(cpi_quarterly)} rows")
print(cpi_quarterly.tail(8).to_string(index=False))

# Viz: CPI + YoY inflation dual axis
fig, ax1 = plt.subplots(figsize=(11, 3.5))
cpi_quarterly['period'] = cpi_quarterly['Year'] + (cpi_quarterly['quarter'] - 1) / 4

ax1.plot(cpi_quarterly['period'], cpi_quarterly['cpi'],
         color='#1F77B4', linewidth=1.8, label='CPI index')
ax1.set_xlabel('Year')
ax1.set_ylabel('CPI index (1982-84 = 100)', color='#1F77B4')
ax1.tick_params(axis='y', labelcolor='#1F77B4')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(cpi_quarterly['period'], cpi_quarterly['cpi_yoy_change'],
         color='#D62728', linewidth=1.2, linestyle='--', label='YoY inflation %')
ax2.axhline(0, color='gray', alpha=0.5, linewidth=0.5)
ax2.set_ylabel('YoY inflation (%)', color='#D62728')
ax2.tick_params(axis='y', labelcolor='#D62728')

plt.title('U.S. CPI-U and Year-over-Year Inflation')
fig.tight_layout()
plt.show()


### 2.5.4 Merge Both Sources into the DOT Primary Table

Both sources join on (Year, quarter) as **left joins onto DOT** — every DOT row is preserved; the external columns are added as new features.

In [ ]:
# Capture pre-merge column set for the before/after view
cols_before_merge = df.columns.tolist()

# Merge 1: EIA fuel (on Year + quarter)
df = df.merge(fuel_quarterly[['Year', 'quarter', 'jet_fuel_usd_per_gal',
                               'jet_fuel_lag_1q', 'jet_fuel_yoy_change']],
              on=['Year', 'quarter'], how='left')

# Merge 2: BLS CPI (on Year + quarter)
df = df.merge(cpi_quarterly[['Year', 'quarter', 'cpi', 'cpi_yoy_change']],
              on=['Year', 'quarter'], how='left')

print("=== MERGE SUMMARY ===")
new_cols = [c for c in df.columns if c not in cols_before_merge]
print(f"Columns before merge: {len(cols_before_merge)}")
print(f"Columns after merge:  {df.shape[1]}")
print(f"New columns: {new_cols}")

print("\n=== JOIN COVERAGE (non-null %) ===")
for col in new_cols:
    pct = df[col].notna().mean() * 100
    print(f"  {col:25s}: {pct:5.1f}%")


### 2.5.5 Handling Missing Values in External Features

EIA fuel and BLS CPI both have ~100% coverage (one value per quarter, all quarters present). The YoY change features have NaN for the first 4 quarters (2010) since no prior-year comparison exists. We handle these:

- **Levels** (fuel price, CPI): forward-fill any gaps (shouldn't be any, but defensive)
- **YoY changes:** fill with 0 (no change) for the first year — reasonable neutral default

In [ ]:
# Level features: forward/back fill (expects full coverage; defensive)
level_cols = ['jet_fuel_usd_per_gal', 'jet_fuel_lag_1q', 'cpi']
df[level_cols] = df[level_cols].ffill().bfill()

# YoY change features: fill with 0 (first year has no comparison quarter)
yoy_cols = ['jet_fuel_yoy_change', 'cpi_yoy_change']
df[yoy_cols] = df[yoy_cols].fillna(0)

print("=== POST-CLEAN NULLS IN NEW COLUMNS ===")
for col in new_cols:
    n = df[col].isna().sum()
    print(f"  {col:25s}: {n}")

print(f"\nFinal shape: {df.shape}")


### 2.5.6 Does the External Data Actually Add Signal?

Two quick sanity-check plots before we move on:

1. **Fare vs fuel price over time** — do they co-move as theory predicts?
2. **Fare vs CPI over time** — does general inflation track airfare levels?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

# Quarterly aggregates for both plots
q_means = df.groupby(['Year', 'quarter']).agg(
    avg_fare=('fare', 'mean'),
    avg_fuel=('jet_fuel_usd_per_gal', 'mean'),
    avg_cpi =('cpi', 'mean')).reset_index()
q_means['period'] = q_means['Year'] + (q_means['quarter'] - 1) / 4

# Plot 1: Fare and fuel on dual axes
ax = axes[0]
ax.plot(q_means['period'], q_means['avg_fare'], color='#1F77B4',
        linewidth=2, label='Avg fare ($)')
ax.set_xlabel('Year')
ax.set_ylabel('Avg Fare ($)', color='#1F77B4')
ax.tick_params(axis='y', labelcolor='#1F77B4')
ax.grid(True, alpha=0.3)

ax2 = ax.twinx()
ax2.plot(q_means['period'], q_means['avg_fuel'], color='#D62728',
         linewidth=1.4, linestyle='--', label='Jet fuel ($/gal)')
ax2.set_ylabel('Jet Fuel ($/gal)', color='#D62728')
ax2.tick_params(axis='y', labelcolor='#D62728')

corr_ff = q_means['avg_fare'].corr(q_means['avg_fuel'])
ax.set_title(f'Avg Fare vs Jet Fuel Price (Pearson r = {corr_ff:.2f})')

# Plot 2: Fare and CPI on dual axes
ax = axes[1]
ax.plot(q_means['period'], q_means['avg_fare'], color='#1F77B4',
        linewidth=2, label='Avg fare ($)')
ax.set_xlabel('Year')
ax.set_ylabel('Avg Fare ($)', color='#1F77B4')
ax.tick_params(axis='y', labelcolor='#1F77B4')
ax.grid(True, alpha=0.3)

ax2 = ax.twinx()
ax2.plot(q_means['period'], q_means['avg_cpi'], color='#2CA02C',
         linewidth=1.4, linestyle='--', label='CPI index')
ax2.set_ylabel('CPI index', color='#2CA02C')
ax2.tick_params(axis='y', labelcolor='#2CA02C')

corr_fc = q_means['avg_fare'].corr(q_means['avg_cpi'])
ax.set_title(f'Avg Fare vs CPI (Pearson r = {corr_fc:.2f})')

plt.tight_layout()
plt.show()

print(f"\nTakeaways:")
print(f"  \u2022 Fare \u2194 fuel correlation: r = {corr_ff:.3f}  "
      f"(positive: fuel cost pass-through)")
print(f"  \u2022 Fare \u2194 CPI correlation:  r = {corr_fc:.3f}  "
      f"(positive: general inflation tracks fare levels)")
print(f"  \u2022 Both signals contribute macro context the DOT-only model lacks.")


### 2.5.7 Summary

We merged two complementary public data sources into the DOT primary table:

| Source | Granularity | New features | Coverage |
|---|---|---|---|
| EIA jet fuel | Quarterly means | `jet_fuel_usd_per_gal`, `jet_fuel_lag_1q`, `jet_fuel_yoy_change` | ~100% |
| BLS CPI | Quarterly means | `cpi`, `cpi_yoy_change` | ~100% |

Total new features: **5**. The rest of the pipeline (cleaning, feature engineering, modeling) continues unchanged, but will now have access to these macroeconomic signals.

In §10.6 we run an **ablation study** comparing DOT-only vs DOT+external model performance to quantify how much these additions actually help — this is the graduate-level rubric's "ablation study" requirement.

**Future work:** Supply-side features from BTS T-100 (load factor, capacity changes) would likely improve predictions further. We deferred this integration due to BTS's session-based download protocol; a clean path forward would be to consume T-100 via a pre-processed mirror on Kaggle or to implement the ASP.NET session handshake.

## 3. Exploratory Data Analysis

Seven visualizations chosen to surface distinct structural properties of the data: target distribution shape, temporal trend, cross-route variation, the distance–fare relationship, seasonality, carrier effects, and feature correlation.

### 3.1 Target Distribution — Fare Histogram (Raw vs Log)

Check whether the target is normally distributed. If skewed, a log-transform may help linear models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['fare'], bins=60, color='steelblue', edgecolor='black')
axes[0].set_title('Fare Distribution (raw $)')
axes[0].set_xlabel('Fare ($)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['fare'].mean(), color='red', ls='--', label=f"mean ${df['fare'].mean():.0f}")
axes[0].axvline(df['fare'].median(), color='orange', ls='--', label=f"median ${df['fare'].median():.0f}")
axes[0].legend()

axes[1].hist(np.log1p(df['fare']), bins=60, color='teal', edgecolor='black')
axes[1].set_title('Fare Distribution (log scale)')
axes[1].set_xlabel('log(1 + Fare)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Skewness (raw):  {df['fare'].skew():.3f}")
print(f"Skewness (log1p): {np.log1p(df['fare']).skew():.3f}")
print("\nInterpretation: the raw fare distribution is right-skewed (long tail of expensive")
print("long-haul routes). The log-transformed distribution is close to symmetric, so we will")
print("train on log1p(fare) for the Ridge baseline to stabilize errors.")


### 3.2 Average Fare Over Time

Tracks the post-2010 era: the recovery from the 2008 recession, LCC maturation, the COVID-19 collapse and rebound.

In [ ]:
yearly = df.groupby('Year')['fare'].agg(['mean', 'std']).reset_index()

plt.figure(figsize=(12, 4.5))
plt.plot(yearly['Year'], yearly['mean'], marker='o', linewidth=2, color='steelblue', label='Mean')
plt.fill_between(yearly['Year'],
                 yearly['mean'] - yearly['std'],
                 yearly['mean'] + yearly['std'],
                 alpha=0.2, color='steelblue', label='±1 SD')
plt.axvspan(2020, 2021, alpha=0.15, color='red', label='COVID era')
plt.title('Average US Domestic Fare by Year (2010–2025)')
plt.xlabel('Year')
plt.ylabel('Fare ($)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"2010 mean fare: ${yearly.iloc[0]['mean']:.2f}")
print(f"2019 mean fare (pre-COVID peak era): ${yearly[yearly['Year']==2019]['mean'].values[0]:.2f}")
print(f"2021 mean fare (COVID trough): ${yearly[yearly['Year']==2021]['mean'].values[0]:.2f}")
print(f"2025 mean fare: ${yearly.iloc[-1]['mean']:.2f}")


### 3.3 Top 10 Busiest Routes

Routes that dominate passenger volume tend to have heavier competition and lower per-mile fares, even over longer distances.

In [ ]:
df['route'] = df['city1'] + ' — ' + df['city2']
top_routes = (df.groupby('route')
                .agg(avg_fare=('fare', 'mean'),
                     avg_pax=('passengers', 'mean'),
                     distance=('nsmiles', 'mean'))
                .nlargest(10, 'avg_pax'))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_routes.index, top_routes['avg_fare'],
               color='teal', edgecolor='black')
ax.set_xlabel('Average Fare ($)')
ax.set_title('Average Fare on the 10 Busiest US Domestic Routes (2010–2025)')
ax.invert_yaxis()
for bar, dist in zip(bars, top_routes['distance']):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{dist:.0f} mi', va='center', fontsize=9)
plt.tight_layout()
plt.show()


### 3.4 Fare vs Distance, Colored by Market Dominance

Each point is one route-quarter. Redder = one carrier dominates, greener = competition. At any given distance, competition routes sit visibly below monopoly routes.

In [ ]:
sample = df.sample(min(20000, len(df)), random_state=RANDOM_STATE)

plt.figure(figsize=(12, 5.5))
sc = plt.scatter(sample['nsmiles'], sample['fare'],
                 c=sample['large_ms'], cmap='RdYlGn_r',
                 alpha=0.25, s=4)
plt.colorbar(sc, label='Largest Carrier Market Share')
plt.title('Fare vs Distance — Colored by Market Dominance (sample of 20K rows)')
plt.xlabel('Distance (miles)')
plt.ylabel('Fare ($)')
plt.tight_layout()
plt.show()

hc = df[df['large_ms'] < 0.5]['fare'].mean()
lc = df[df['large_ms'] >= 0.5]['fare'].mean()
print(f"High competition (largest-carrier share < 50%) mean fare: ${hc:.2f}")
print(f"Low competition  (largest-carrier share ≥ 50%) mean fare: ${lc:.2f}")
print(f"Competition discount: ${lc - hc:.2f} ({(lc-hc)/lc*100:.1f}%)")


### 3.5 Quarterly Seasonality

Are there systematic fare differences across quarters (holiday travel, summer vacation)?

In [ ]:
plt.figure(figsize=(9, 4.5))
sns.boxplot(data=df, x='quarter', y='fare', showfliers=False,
            palette='Blues')
plt.title('Fare Distribution by Quarter (outliers hidden)')
plt.xlabel('Quarter')
plt.ylabel('Fare ($)')
plt.tight_layout()
plt.show()

q_means = df.groupby('quarter')['fare'].mean()
print("Quarterly mean fares:")
for q, v in q_means.items():
    print(f"  Q{q}: ${v:.2f}")
print(f"\nRange: ${q_means.max() - q_means.min():.2f}  "
      f"(weakly seasonal, but worth including as a feature)")


### 3.6 Fare by Carrier

Legacy carriers (AA, DL, UA) typically command a premium over LCCs (WN, NK, F9, G4).

In [ ]:
top_carriers = df['carrier_lg'].value_counts().head(10).index
carrier_data = df[df['carrier_lg'].isin(top_carriers)]

plt.figure(figsize=(11, 4.5))
sns.boxplot(data=carrier_data, x='carrier_lg', y='fare',
            order=top_carriers, showfliers=False, palette='Set3')
plt.title('Fare Distribution by Largest Carrier (top 10 by frequency)')
plt.xlabel('Largest Carrier (IATA code)')
plt.ylabel('Fare ($)')
plt.tight_layout()
plt.show()

print("Median fare by carrier:")
print(df.groupby('carrier_lg')['fare'].median().sort_values(ascending=False).head(10).round(2))


### 3.7 Correlation Heatmap — All Numeric Features

Pairwise Pearson correlations among all numeric features (DOT + external) and the target. Surfaces redundancy (e.g., fuel-level vs lagged fuel), multicollinearity concerns, and which individual features have strong marginal associations with fare.

In [ ]:
# Include both DOT numeric features AND external (fuel + CPI) features
num_cols = [
    # DOT
    'Year', 'quarter', 'nsmiles', 'passengers', 'large_ms',
    # External (from §2.5)
    'jet_fuel_usd_per_gal', 'jet_fuel_lag_1q', 'jet_fuel_yoy_change',
    'cpi', 'cpi_yoy_change',
    # Target
    'fare',
]

# Defensive: some external columns might not exist if §2.5 didn't run
num_cols = [c for c in num_cols if c in df.columns]
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, cbar_kws={'label': 'Pearson r'},
            annot_kws={'size': 9})
plt.title('Pearson Correlation \u2014 All Numeric Features (DOT + External) vs Fare')
plt.tight_layout()
plt.show()

# Interpretation: rank features by absolute correlation with fare
print("\n=== Features ranked by |correlation with fare| ===")
fare_corr = corr['fare'].drop('fare').abs().sort_values(ascending=False)
for feat, val in fare_corr.items():
    signed = corr.loc[feat, 'fare']
    direction = '+' if signed > 0 else '-'
    print(f"  {feat:25s}: r = {signed:+.3f}  (|r| = {val:.3f}) {direction}")

print("\n=== Redundancy check (|r| > 0.8 between feature pairs) ===")
feats_only = [c for c in num_cols if c != 'fare']
redundant_pairs = []
for i, a in enumerate(feats_only):
    for b in feats_only[i+1:]:
        r = corr.loc[a, b]
        if abs(r) > 0.8:
            redundant_pairs.append((a, b, r))
if redundant_pairs:
    for a, b, r in redundant_pairs:
        print(f"  {a} \u2194 {b}: r = {r:+.3f}  (highly correlated, possible redundancy)")
else:
    print("  No feature pairs above |r| > 0.8  (no severe redundancy)")


## 4. Feature Engineering

**Eight derived features from the DOT primary data** + **five passthrough external features** from the §2.5 augmentation step. All 13 are pre-outcome (no target leakage).

**Derived from DOT:**

| # | Feature | Rationale |
|---|---------|-----------|
| 1 | `log_distance` | Fare-vs-distance is concave (bulk discount on long hauls) — log helps linear models |
| 2 | `log_passengers` | Passenger volume is right-skewed; log stabilizes variance |
| 3 | `is_long_haul` | Binary flag for > 1,500 mi — captures non-linear jump in fare |
| 4 | `distance_bucket` | Ordinal: short / medium / long / transcon |
| 5 | `quarter_sin`, `quarter_cos` | Cyclical encoding of quarter (Q4 and Q1 are "close" seasonally) |
| 6 | `is_covid_era` | 2020—2021 indicator — major market disruption |
| 7 | `carrier_tier` | Legacy / LCC / ULCC / Regional — compresses 20+ carrier codes into 4 tiers |
| 8 | `route_popularity` | Mean passengers on this route (computed train-only later to avoid leakage) |

**Passthrough from external sources (created in §2.5):**

| # | Feature | Source | Rationale |
|---|---------|--------|-----------|
| 9 | `jet_fuel_usd_per_gal` | EIA | Direct fuel cost signal |
| 10 | `jet_fuel_lag_1q` | EIA | Captures lagged fuel pass-through (hedging + adjustment delay) |
| 11 | `jet_fuel_yoy_change` | EIA | Fuel shock indicator |
| 12 | `cpi` | BLS | General price level |
| 13 | `cpi_yoy_change` | BLS | Inflation rate signal |

Total features the model will see: **13 engineered + a few passthrough DOT columns (Year, quarter, nsmiles, passengers, large_ms) + categorical encodings = 22 total** (finalized in §5).

In [ ]:
# 1 & 2. Log transforms
df['log_distance'] = np.log1p(df['nsmiles'])
df['log_passengers'] = np.log1p(df['passengers'])

# 3. Long haul flag
df['is_long_haul'] = (df['nsmiles'] > 1500).astype(int)

# 4. Distance bucket (ordinal)
df['distance_bucket'] = pd.cut(df['nsmiles'],
                                bins=[0, 500, 1000, 1500, np.inf],
                                labels=[0, 1, 2, 3]).astype(int)

# 5. Cyclical quarter encoding
df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)

# 6. COVID era indicator
df['is_covid_era'] = df['Year'].isin([2020, 2021]).astype(int)

# 7. Carrier tier
LEGACY = {'AA', 'DL', 'UA', 'US', 'CO', 'NW', 'HP'}
LCC    = {'WN', 'B6', 'AS', 'HA', 'SY', 'VX'}
ULCC   = {'NK', 'F9', 'G4', 'MX'}
def carrier_tier(code):
    if code in LEGACY: return 0
    if code in LCC:    return 1
    if code in ULCC:   return 2
    return 3  # regional / other
df['carrier_tier'] = df['carrier_lg'].apply(carrier_tier)

# 8. route_popularity will be computed after split (train-only) to avoid leakage
df['route_popularity'] = 0.0  # placeholder, overwritten after split

# --- Derived features from DOT ---
eng_dot = ['log_distance', 'log_passengers', 'is_long_haul',
           'distance_bucket', 'quarter_sin', 'quarter_cos',
           'is_covid_era', 'carrier_tier']

# --- Passthrough features from §2.5 external augmentation ---
eng_external = ['jet_fuel_usd_per_gal', 'jet_fuel_lag_1q', 'jet_fuel_yoy_change',
                'cpi', 'cpi_yoy_change']

print("=== DOT-DERIVED FEATURES ===")
print(df[eng_dot].describe().round(3).to_string())
print(f"\nDistance bucket distribution: "
      f"{df['distance_bucket'].value_counts().sort_index().to_dict()}")
print(f"Carrier tier distribution:    "
      f"{df['carrier_tier'].value_counts().sort_index().to_dict()}")
print(f"COVID-era rows:               {df['is_covid_era'].sum():,}")

print("\n=== EXTERNAL PASSTHROUGH FEATURES (from §2.5) ===")
missing_ext = [c for c in eng_external if c not in df.columns]
if missing_ext:
    print(f"WARNING: external features missing from df: {missing_ext}")
    print("Did §2.5 run successfully? Re-run §2.5 before continuing.")
else:
    print(df[eng_external].describe().round(3).to_string())

print(f"\nTotal engineered features: {len(eng_dot) + len(eng_external)} "
      f"({len(eng_dot)} DOT-derived + {len(eng_external)} external passthrough)")


## 5. Preprocessing for Modeling

Encode the remaining categorical columns. Cities have 90+ categories — too many for one-hot — so we use frequency encoding. `carrier_lg` gets label-encoded.

In [ ]:
# Encode carrier_lg
le_carrier = LabelEncoder()
df['carrier_lg_enc'] = le_carrier.fit_transform(df['carrier_lg'])

# Frequency-encode cities
city1_freq = df['city1'].value_counts() / len(df)
city2_freq = df['city2'].value_counts() / len(df)
df['city1_freq'] = df['city1'].map(city1_freq)
df['city2_freq'] = df['city2'].map(city2_freq)

# Final feature set
FEATURE_COLS = [
    # DOT passthrough
    'Year', 'quarter', 'nsmiles', 'passengers', 'large_ms',
    # DOT-derived engineered
    'log_distance', 'log_passengers',
    'is_long_haul', 'distance_bucket',
    'quarter_sin', 'quarter_cos', 'is_covid_era',
    'carrier_tier', 'carrier_lg_enc',
    'city1_freq', 'city2_freq',
    'route_popularity',  # filled in after split
    # External passthrough (from §2.5)
    'jet_fuel_usd_per_gal', 'jet_fuel_lag_1q', 'jet_fuel_yoy_change',
    'cpi', 'cpi_yoy_change',
]
TARGET = 'fare'

# Safety check: all columns present
missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    raise RuntimeError(
        f"FEATURE_COLS references missing columns: {missing}. "
        f"Did §2.5 (external augmentation) and §4 (feature engineering) both run?")

X = df[FEATURE_COLS + ['city1', 'city2']].copy()  # keep route keys for leakage-safe pop calc
y = df[TARGET].copy()
print(f"Feature matrix: {X.shape}")
print(f"Target: {y.shape}")
print(f"\nTotal features: {len(FEATURE_COLS)}")
print(f"Features: {FEATURE_COLS}")


## 6. Train / Validation / Test Split (60 / 20 / 20)

- **Train (60%)** — fit models, compute route-popularity statistics
- **Validation (20%)** — hyperparameter tuning
- **Test (20%)** — held out until the very end, reported once

After splitting, we compute `route_popularity` using the **training set only** and map it back to all three splits. Computing it on the full dataset would leak target-adjacent information (high-volume routes correlate with certain fare regimes).

In [ ]:
# First split off the test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE)

# Then split train/val from the remaining 80%
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=RANDOM_STATE)
# 0.25 * 0.80 = 0.20, so final ratio is 60/20/20

print(f"Train: {X_train.shape[0]:,} rows")
print(f"Val:   {X_val.shape[0]:,} rows")
print(f"Test:  {X_test.shape[0]:,} rows")

# Compute route_popularity on TRAIN ONLY
route_pop = (X_train.assign(pax=y_train)  # dummy — just need passengers
                    .groupby(['city1', 'city2'])['pax']
                    .transform('count'))  # count train observations per route
# But we want a route-level mapping, so:
route_pop_map = (X_train.groupby(['city1', 'city2']).size() / len(X_train)).to_dict()
global_mean_pop = np.mean(list(route_pop_map.values()))

def map_route_pop(df_part):
    return df_part.apply(lambda r: route_pop_map.get((r['city1'], r['city2']),
                                                      global_mean_pop), axis=1)

X_train['route_popularity'] = map_route_pop(X_train)
X_val['route_popularity']   = map_route_pop(X_val)
X_test['route_popularity']  = map_route_pop(X_test)

# Drop the route-key columns (not model features)
X_train = X_train[FEATURE_COLS]
X_val   = X_val[FEATURE_COLS]
X_test  = X_test[FEATURE_COLS]

print(f"\nRoute popularity range: [{X_train['route_popularity'].min():.6f}, "
      f"{X_train['route_popularity'].max():.6f}]")


## 7. Three Baseline Models from Three Different Families

| Model | Family | Why it's a good baseline |
|-------|--------|--------------------------|
| **Ridge Regression** | Linear (L2-regularized) | Fast sanity check; tells us how much non-linear structure exists |
| **Random Forest** | Tree ensemble — bagging | Handles non-linearity + interactions with no scaling needed |
| **HistGradientBoosting** | Tree ensemble — boosting | Typically state-of-the-art on tabular regression |

All three are evaluated with **5-fold cross-validation on the training set** (not just train/val split) to get confidence-interval-style variance.

In [ ]:
def eval_model(name, model, X_tr, y_tr, X_v, y_v):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_v)
    return {
        'Model': name,
        'R2':   r2_score(y_v, pred),
        'MAE':  mean_absolute_error(y_v, pred),
        'RMSE': np.sqrt(mean_squared_error(y_v, pred)),
        'MAPE': np.mean(np.abs((y_v - pred) / y_v)) * 100,
    }

def cv_score(model, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(model, X, y, cv=kf,
                             scoring='neg_mean_absolute_error', n_jobs=-1)
    return -scores  # convert to positive MAE


### 7.1 Ridge Regression (linear family)

In [ ]:
# Ridge needs scaled features
ridge_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('ridge', Ridge(alpha=1.0, random_state=RANDOM_STATE))
])

ridge_cv = cv_score(ridge_pipe, X_train, y_train)
print(f"Ridge 5-fold CV MAE: ${ridge_cv.mean():.2f} ± ${ridge_cv.std():.2f}")

ridge_val = eval_model('Ridge', ridge_pipe, X_train, y_train, X_val, y_val)
print(f"\nRidge on validation:")
for k, v in ridge_val.items():
    print(f"  {k}: {v if isinstance(v,str) else f'{v:.4f}' if k=='R2' else f'{v:.2f}'}")


### 7.2 Random Forest (bagging ensemble)

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=None,
                            min_samples_leaf=2, n_jobs=-1,
                            random_state=RANDOM_STATE)

rf_cv = cv_score(rf, X_train, y_train)
print(f"Random Forest 5-fold CV MAE: ${rf_cv.mean():.2f} ± ${rf_cv.std():.2f}")

rf_val = eval_model('Random Forest', rf, X_train, y_train, X_val, y_val)
print(f"\nRandom Forest on validation:")
for k, v in rf_val.items():
    print(f"  {k}: {v if isinstance(v,str) else f'{v:.4f}' if k=='R2' else f'{v:.2f}'}")


### 7.3 HistGradientBoosting (boosting ensemble)

In [ ]:
hgb = HistGradientBoostingRegressor(
    learning_rate=0.1, max_iter=300, max_depth=None,
    max_leaf_nodes=31, min_samples_leaf=20,
    random_state=RANDOM_STATE)

hgb_cv = cv_score(hgb, X_train, y_train)
print(f"HistGradientBoosting 5-fold CV MAE: ${hgb_cv.mean():.2f} ± ${hgb_cv.std():.2f}")

hgb_val = eval_model('HistGradientBoosting', hgb, X_train, y_train, X_val, y_val)
print(f"\nHistGradientBoosting on validation:")
for k, v in hgb_val.items():
    print(f"  {k}: {v if isinstance(v,str) else f'{v:.4f}' if k=='R2' else f'{v:.2f}'}")


### 7.4 Baseline Comparison Table

In [ ]:
baseline_df = pd.DataFrame([ridge_val, rf_val, hgb_val])
baseline_df['CV_MAE_mean'] = [ridge_cv.mean(), rf_cv.mean(), hgb_cv.mean()]
baseline_df['CV_MAE_std']  = [ridge_cv.std(),  rf_cv.std(),  hgb_cv.std()]
baseline_df = baseline_df[['Model', 'R2', 'MAE', 'RMSE', 'MAPE',
                           'CV_MAE_mean', 'CV_MAE_std']]

print("=== BASELINE MODEL COMPARISON ===")
print(baseline_df.round(3).to_string(index=False))

best_baseline = baseline_df.loc[baseline_df['MAE'].idxmin(), 'Model']
print(f"\nBest baseline by validation MAE: {best_baseline}")


## 8. Hyperparameter Tuning — RandomizedSearchCV

Tune the best baseline (HistGradientBoosting). Random search with 20 iterations × 5-fold CV = 100 fits.

In [ ]:
param_dist = {
    'learning_rate':    [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    'max_iter':         [200, 300, 500, 800],
    'max_leaf_nodes':   [15, 31, 63, 127],
    'min_samples_leaf': [10, 20, 50, 100],
    'l2_regularization':[0.0, 0.1, 1.0, 10.0],
}

search = RandomizedSearchCV(
    estimator=HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    param_distributions=param_dist,
    n_iter=20,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)

search.fit(X_train, y_train)

print(f"\nBest CV MAE: ${-search.best_score_:.2f}")
print(f"Best hyperparameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

tuned_model = search.best_estimator_
tuned_val = eval_model('Tuned HGB', tuned_model, X_train, y_train, X_val, y_val)
print(f"\nTuned HGB on validation:")
for k, v in tuned_val.items():
    print(f"  {k}: {v if isinstance(v,str) else f'{v:.4f}' if k=='R2' else f'{v:.2f}'}")


## 9. Feature Selection — Permutation Importance

Unlike impurity-based importance, permutation importance measures the *causal* drop in validation R² when a feature is shuffled. Features with near-zero permutation importance are candidates for removal.

In [ ]:
perm = permutation_importance(tuned_model, X_val, y_val,
                               n_repeats=5, random_state=RANDOM_STATE,
                               n_jobs=-1, scoring='neg_mean_absolute_error')

perm_df = pd.DataFrame({
    'Feature': X_val.columns,
    'Importance_mean': perm.importances_mean,
    'Importance_std':  perm.importances_std,
}).sort_values('Importance_mean', ascending=False)

print("=== PERMUTATION IMPORTANCE (drop in MAE when feature is shuffled) ===")
print(perm_df.round(4).to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(perm_df['Feature'][::-1], perm_df['Importance_mean'][::-1],
         xerr=perm_df['Importance_std'][::-1], color='steelblue')
plt.xlabel('Permutation Importance (Δ MAE)')
plt.title('Feature Importance — Tuned HistGradientBoosting')
plt.tight_layout()
plt.show()

low_imp = perm_df[perm_df['Importance_mean'] < 0.5]['Feature'].tolist()
print(f"\nFeatures with low importance (< $0.50 MAE contribution): {low_imp}")
print("Per the permutation study, these could be pruned without meaningful loss.")


## 10. Final Evaluation on Held-Out Test Set

Evaluate the tuned model **once** on the test set. This is the number we report in the paper.

In [ ]:
# Retrain tuned model on train + val combined (standard practice before final test)
X_trval = pd.concat([X_train, X_val])
y_trval = pd.concat([y_train, y_val])
final_model = HistGradientBoostingRegressor(**search.best_params_,
                                             random_state=RANDOM_STATE)
final_model.fit(X_trval, y_trval)

y_pred_test = final_model.predict(X_test)

test_metrics = {
    'R2':   r2_score(y_test, y_pred_test),
    'MAE':  mean_absolute_error(y_test, y_pred_test),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
    'MAPE': np.mean(np.abs((y_test - y_pred_test) / y_test)) * 100,
}

print("=== FINAL TEST SET PERFORMANCE ===")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}" if k == 'R2' else f"  {k}: {v:.3f}")

# Residual analysis
residuals = y_test - y_pred_test
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(y_pred_test, y_test, alpha=0.15, s=4, color='steelblue')
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
axes[0].plot(lims, lims, 'r--', lw=1)
axes[0].set_xlabel('Predicted Fare ($)')
axes[0].set_ylabel('Actual Fare ($)')
axes[0].set_title('Predicted vs Actual (Test Set)')

axes[1].hist(residuals, bins=50, color='teal', edgecolor='black')
axes[1].axvline(0, color='red', ls='--')
axes[1].set_xlabel('Residual ($ actual − predicted)')
axes[1].set_title(f'Residual Distribution  (mean = ${residuals.mean():.2f})')

plt.tight_layout()
plt.show()

print(f"\nResidual analysis:")
print(f"  Mean residual: ${residuals.mean():.2f}  (should be ≈ 0 → unbiased)")
print(f"  Std residual:  ${residuals.std():.2f}")
print(f"  90% of predictions are within ±${np.percentile(np.abs(residuals), 90):.2f}")


## 10.5 Temporal Holdout — Can This Model Forecast?

The main evaluation above uses a **random** train/test split, so the test set contains rows from the same years as training. That measures *estimation* accuracy — can the model fill in an unseen route-carrier-quarter when it has seen many similar ones from the same era?

**Forecasting** is a different question: can the model predict years it has *never seen*? We test this by training on 2010–2023 and evaluating on 2024–2025.

This is the project's honest failure-case analysis. The point is to quantify the gap between what the model can and can't do, not to pretend it can forecast.

In [ ]:
# Build a temporal split: train on 2010-2023, test on 2024-2025
temporal_train_mask = df['Year'] <= 2023
temporal_test_mask  = df['Year'] >= 2024

df_ttrain = df[temporal_train_mask].copy()
df_ttest  = df[temporal_test_mask].copy()

print(f"Temporal train (2010-2023): {len(df_ttrain):,} rows")
print(f"Temporal test  (2024-2025): {len(df_ttest):,} rows")

# Recompute route_popularity using temporal train ONLY
temp_route_pop_map = (df_ttrain.groupby(['city1', 'city2']).size() / len(df_ttrain)).to_dict()
temp_global_mean = np.mean(list(temp_route_pop_map.values()))

def map_temp_pop(frame):
    return frame.apply(lambda r: temp_route_pop_map.get((r['city1'], r['city2']),
                                                         temp_global_mean), axis=1)

df_ttrain['route_popularity'] = map_temp_pop(df_ttrain)
df_ttest['route_popularity']  = map_temp_pop(df_ttest)

X_ttrain = df_ttrain[FEATURE_COLS]
y_ttrain = df_ttrain[TARGET]
X_ttest  = df_ttest[FEATURE_COLS]
y_ttest  = df_ttest[TARGET]

print(f"\nFeature cols: {X_ttrain.shape[1]}")


### 10.5.1 Train the Same Tuned Model on Pre-2024 Data Only

In [ ]:
# Train tuned-hyperparameter HGB on 2010-2023 only, test on 2024-2025
temporal_model = HistGradientBoostingRegressor(**search.best_params_,
                                                random_state=RANDOM_STATE)
temporal_model.fit(X_ttrain, y_ttrain)

y_tpred = temporal_model.predict(X_ttest)

temporal_metrics = {
    'R2':   r2_score(y_ttest, y_tpred),
    'MAE':  mean_absolute_error(y_ttest, y_tpred),
    'RMSE': np.sqrt(mean_squared_error(y_ttest, y_tpred)),
    'MAPE': np.mean(np.abs((y_ttest - y_tpred) / y_ttest)) * 100,
}

print("=== TEMPORAL HOLDOUT (train <=2023, test 2024-2025) ===")
for k, v in temporal_metrics.items():
    fmt = '.4f' if k == 'R2' else '.3f'
    print(f"  {k}: {v:{fmt}}")

print("\n=== RANDOM SPLIT (from section 10, for comparison) ===")
print(f"  R2:   {test_metrics['R2']:.4f}")
print(f"  MAE:  {test_metrics['MAE']:.3f}")
print(f"  RMSE: {test_metrics['RMSE']:.3f}")
print(f"  MAPE: {test_metrics['MAPE']:.3f}")

print("\n=== DEGRADATION: forecasting vs estimation ===")
print(f"  R2 drops from {test_metrics['R2']:.3f} to {temporal_metrics['R2']:.3f}")
print(f"  MAE rises from ${test_metrics['MAE']:.2f} to ${temporal_metrics['MAE']:.2f}")
print(f"  MAPE rises from {test_metrics['MAPE']:.2f}% to {temporal_metrics['MAPE']:.2f}%")


### 10.5.2 Visualizing the Forecasting Failure

Three plots tell the story:

1. **Yearly actual vs predicted** - the model's predicted trend flattens after its training cutoff
2. **Residuals by year** - systematic bias in 2024-2025 (not just noise)
3. **Random vs temporal split metrics** - side-by-side degradation

In [ ]:
# Plot 1: Yearly mean actual vs predicted
df_ttest_plot = df_ttest[['Year']].copy()
df_ttest_plot['actual']    = y_ttest.values
df_ttest_plot['predicted'] = y_tpred

df_ttrain_plot = df_ttrain[['Year']].copy()
df_ttrain_plot['actual']    = y_ttrain.values
df_ttrain_plot['predicted'] = temporal_model.predict(X_ttrain)

combined = pd.concat([df_ttrain_plot, df_ttest_plot])
yearly_means = combined.groupby('Year').agg(actual=('actual', 'mean'),
                                             predicted=('predicted', 'mean'))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

ax = axes[0]
ax.plot(yearly_means.index, yearly_means['actual'], 'o-',
        label='Actual', color='black', linewidth=2)
ax.plot(yearly_means.index, yearly_means['predicted'], 's--',
        label='Model prediction', color='#D62728', linewidth=2)
ax.axvspan(2023.5, 2025.5, alpha=0.15, color='red', label='Out-of-sample years')
ax.set_xlabel('Year')
ax.set_ylabel('Mean Fare ($)')
ax.set_title('Actual vs Predicted Mean Fare by Year')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 2: Residuals by year
ax = axes[1]
combined['residual'] = combined['actual'] - combined['predicted']
for year in sorted(combined['Year'].unique()):
    res = combined[combined['Year'] == year]['residual']
    color = '#D62728' if year >= 2024 else '#1F77B4'
    ax.scatter([year] * len(res), res, alpha=0.05, s=2, color=color)

yearly_res = combined.groupby('Year')['residual'].mean()
ax.plot(yearly_res.index, yearly_res.values, 'o-', color='black',
        linewidth=2, label='Mean residual', zorder=10)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvspan(2023.5, 2025.5, alpha=0.15, color='red')
ax.set_xlabel('Year')
ax.set_ylabel('Residual (actual - predicted, $)')
ax.set_title('Residuals by Year: Systematic Bias Out-of-Sample')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 3: Metric comparison bar chart
ax = axes[2]
labels = ['R2 (x100)', 'MAE ($)', 'MAPE (%)']
random_vals   = [test_metrics['R2'] * 100,     test_metrics['MAE'],     test_metrics['MAPE']]
temporal_vals = [temporal_metrics['R2'] * 100, temporal_metrics['MAE'], temporal_metrics['MAPE']]

x = np.arange(len(labels))
w = 0.35
ax.bar(x - w/2, random_vals,   w, label='Random split (estimation)',    color='#1F77B4')
ax.bar(x + w/2, temporal_vals, w, label='Temporal split (forecasting)', color='#D62728')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Value')
ax.set_title('Estimation vs Forecasting Performance')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - On a random split (estimation), the model interpolates within known years well.")
print("  - On a temporal split (true forecasting), MAE roughly doubles and residuals")
print("    show systematic bias - the model under-predicts 2024-2025 fares because it")
print("    cannot extrapolate beyond its training range.")
print("  - This is a fundamental limitation of tree-based models. For true forecasting")
print("    we would need a linear-trend model, a time-series model (ARIMA/Prophet), or")
print("    a temporal transformer - listed as future work.")


### 10.5.3 Takeaway

| Use case | Appropriate? |
|----------|--------------|
| "What's the typical fare for a Delta Chicago→LAX Q2 flight in 2022?" | Yes — estimation within known years |
| "What's the typical fare for a route I don't see in the data?" | With caution — model defaults to average-route behavior |
| "What will fares look like in 2027?" | No — the model cannot extrapolate |

We deploy the model as a **fare estimator** (§13), with the year input capped at the training range to prevent users from misinterpreting flatlined extrapolations as forecasts.

## 10.6 Ablation Study — How Much Do External Features Help?

The model we evaluated in §10 uses all 22 features, including the 5 we added from EIA (fuel) and BLS (CPI) in §2.5. But how much of our accuracy actually comes from those external features, versus just the original DOT data?

We answer this with a **controlled ablation**: train the same tuned HistGradientBoosting model twice — once on DOT-only features, once on DOT + external — and compare test-set performance. Everything else (split, hyperparameters, random seed) is held constant.

This directly satisfies the graduate-level rubric requirement for an ablation study, and gives us concrete evidence of whether the external data integration was worth the effort.

In [ ]:
# Define two feature sets: DOT-only vs full (DOT + external)
EXTERNAL_COLS = ['jet_fuel_usd_per_gal', 'jet_fuel_lag_1q', 'jet_fuel_yoy_change',
                 'cpi', 'cpi_yoy_change']
DOT_ONLY_COLS = [c for c in FEATURE_COLS if c not in EXTERNAL_COLS]

print(f"DOT-only features:  {len(DOT_ONLY_COLS)}")
print(f"Full (DOT + ext):   {len(FEATURE_COLS)}")

# Use the same train/val/test split as §6, same tuned hyperparameters as §8
# Train both models on train+val, test on held-out test set
X_trval = pd.concat([X_train, X_val])
y_trval = pd.concat([y_train, y_val])

def eval_with_features(feat_cols, label):
    model = HistGradientBoostingRegressor(**search.best_params_,
                                           random_state=RANDOM_STATE)
    model.fit(X_trval[feat_cols], y_trval)
    pred = model.predict(X_test[feat_cols])
    return {
        'setup': label,
        'n_features': len(feat_cols),
        'R2':   r2_score(y_test, pred),
        'MAE':  mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAPE': np.mean(np.abs((y_test - pred) / y_test)) * 100,
    }

dot_only_metrics = eval_with_features(DOT_ONLY_COLS, 'DOT-only')
full_metrics     = eval_with_features(FEATURE_COLS, 'DOT + external')

ablation_df = pd.DataFrame([dot_only_metrics, full_metrics])
print("\n=== ABLATION: DOT-only vs DOT+external ===")
print(ablation_df.round(4).to_string(index=False))

# Compute the lift
mae_improvement  = dot_only_metrics['MAE']  - full_metrics['MAE']
rmse_improvement = dot_only_metrics['RMSE'] - full_metrics['RMSE']
r2_improvement   = full_metrics['R2']       - dot_only_metrics['R2']

print(f"\n=== LIFT FROM EXTERNAL DATA ===")
print(f"  MAE improvement:  ${mae_improvement:+.3f}  "
      f"({mae_improvement/dot_only_metrics['MAE']*100:+.2f}% relative)")
print(f"  RMSE improvement: ${rmse_improvement:+.3f}  "
      f"({rmse_improvement/dot_only_metrics['RMSE']*100:+.2f}% relative)")
print(f"  R2 improvement:   {r2_improvement:+.4f}")


### 10.6.1 Visual Comparison

In [ ]:
# Side-by-side bar comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics_to_plot = [('MAE', '$', '#1F77B4'),
                   ('RMSE', '$', '#2CA02C'),
                   ('MAPE', '%', '#D62728')]

for ax, (metric, unit, color) in zip(axes, metrics_to_plot):
    vals = [dot_only_metrics[metric], full_metrics[metric]]
    labels = ['DOT-only', 'DOT + external']
    bars = ax.bar(labels, vals, color=['#9ca3af', color], edgecolor='black')
    ax.set_title(f'{metric} ({unit})')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f'{unit if unit == "$" else ""}{v:.2f}{unit if unit == "%" else ""}',
                ha='center', fontsize=11, fontweight='bold')
    # Show % improvement as annotation
    improvement = (vals[0] - vals[1]) / vals[0] * 100
    ax.set_xlabel(f'{improvement:+.1f}% with external features',
                  fontsize=9, color='gray')

plt.suptitle('Ablation: Effect of External Data on Test-Set Error', y=1.02)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
if mae_improvement > 0:
    print(f"  External features reduce MAE by ${mae_improvement:.2f} "
          f"({mae_improvement/dot_only_metrics['MAE']*100:.1f}% relative).")
    print(f"  This confirms that jet fuel prices and CPI contribute signal")
    print(f"  beyond what the DOT dataset alone captures.")
elif mae_improvement < -0.5:
    print(f"  External features INCREASE MAE by ${-mae_improvement:.2f} "
          f"({-mae_improvement/dot_only_metrics['MAE']*100:.1f}% relative).")
    print(f"  Worth investigating: the external features may be redundant")
    print(f"  with DOT signals (Year acts as a proxy for both fuel and CPI trends).")
else:
    print(f"  External features have essentially no effect on MAE ({mae_improvement:+.2f}).")
    print(f"  The model already captures equivalent information via Year and DOT-derived")
    print(f"  features. External data adds rigor but not predictive power for this setup.")


### 10.6.2 Discussion

Whatever the sign and magnitude of the improvement, this ablation is **the honest answer** to the question "was the external data integration worth it?" Three possible outcomes and what each means:

| Outcome | Interpretation |
|---|---|
| **External features reduce MAE significantly (>5%)** | Macroeconomic signals carry information the model couldn't otherwise learn. The integration paid off. |
| **Negligible change (±1-2%)** | Tree models may have already learned equivalent information via `Year` (which correlates with both fuel price trends and CPI). External features are still defensible for interpretability. |
| **External features hurt** | Overfitting on the external features, or redundancy causing noise. Worth dropping or regularizing. |

Whichever we observe, reporting it honestly is what distinguishes rigorous evaluation from a just-so story. In the final report, we note the outcome explicitly in the Discussion section.

## 11. Final Model Comparison Table

In [ ]:
final_df = baseline_df[['Model', 'R2', 'MAE', 'RMSE', 'MAPE']].copy()
final_df = pd.concat([final_df, pd.DataFrame([{
    'Model': 'Tuned HGB (final)',
    'R2':    test_metrics['R2'],
    'MAE':   test_metrics['MAE'],
    'RMSE':  test_metrics['RMSE'],
    'MAPE':  test_metrics['MAPE'],
}])], ignore_index=True)

print("=== ALL MODELS — FINAL COMPARISON ===")
print("(Baselines evaluated on validation set; Tuned HGB evaluated on test set)")
print(final_df.round(3).to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4.5))
models = final_df['Model'].values
mae = final_df['MAE'].values
bars = ax.bar(models, mae, color=['#D62728', '#1F77B4', '#2CA02C', '#FF7F0E'])
ax.set_ylabel('MAE ($)')
ax.set_title('Mean Absolute Error by Model')
for bar, v in zip(bars, mae):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'${v:.2f}', ha='center', fontsize=10)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 12. Save Model & Lookups for Deployment

In [ ]:
# Fit final model on ALL data (train+val+test) for deployment
X_all = pd.concat([X_train, X_val, X_test])
y_all = pd.concat([y_train, y_val, y_test])
deploy_model = HistGradientBoostingRegressor(**search.best_params_,
                                              random_state=RANDOM_STATE)
deploy_model.fit(X_all, y_all)

# Build (Year, quarter) -> macro feature lookups for the Gradio app.
# These come from the augmented df that includes §2.5 external columns.
macro_lookup = {}
macro_cols = ['jet_fuel_usd_per_gal', 'jet_fuel_lag_1q', 'jet_fuel_yoy_change',
              'cpi', 'cpi_yoy_change']
for (yr, q), group in df.groupby(['Year', 'quarter']):
    macro_lookup[(int(yr), int(q))] = {col: float(group[col].iloc[0])
                                        for col in macro_cols}

# Fallback: most-recent quarter values, for when user selects a year/quarter
# the lookup doesn't have (shouldn't happen with year slider capped at 2025,
# but defensive)
_recent_key = max(macro_lookup.keys())
macro_defaults = macro_lookup[_recent_key]

print(f"Built macro lookup for {len(macro_lookup)} (year, quarter) combinations")
print(f"Most recent available: {_recent_key}  values: {macro_defaults}")

# Build deployment artifacts
artifacts = {
    'model': deploy_model,
    'feature_cols': FEATURE_COLS,
    'carrier_encoder': le_carrier,
    'city1_freq': city1_freq.to_dict(),
    'city2_freq': city2_freq.to_dict(),
    'route_pop_map': route_pop_map,
    'global_mean_pop': global_mean_pop,
    # Helper lookups for the UI
    'distance_lookup':  df.groupby(['city1', 'city2'])['nsmiles'].mean().to_dict(),
    'passengers_lookup':df.groupby(['city1', 'city2'])['passengers'].mean().to_dict(),
    'ms_lookup':        df.groupby(['city1', 'city2'])['large_ms'].mean().to_dict(),
    # Macro lookups (NEW — needed for fuel + CPI features at inference time)
    'macro_lookup':     macro_lookup,
    'macro_defaults':   macro_defaults,
    'macro_cols':       macro_cols,
    'cities':           sorted(df['city1'].unique().tolist()),
    'carriers':         sorted(df['carrier_lg'].unique().tolist()),
    'legacy_set': LEGACY, 'lcc_set': LCC, 'ulcc_set': ULCC,
}

with open('us_flight_fare_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print(f"\nSaved artifacts to us_flight_fare_artifacts.pkl")
print(f"File size: {__import__('os').path.getsize('us_flight_fare_artifacts.pkl') / 1024 / 1024:.1f} MB")


## 13. Gradio Deployment — Interactive Fare **Estimator**

Year slider is capped at the training range (2010–2025) because the temporal-holdout analysis in §10.5 showed the model cannot extrapolate. The app is framed as a historical-pattern **estimator**, which is what the model actually does well.

In [ ]:
import gradio as gr

with open('us_flight_fare_artifacts.pkl', 'rb') as f:
    art = pickle.load(f)

CARRIER_NAMES = {
    'AA': 'American Airlines', 'DL': 'Delta Air Lines', 'UA': 'United Airlines',
    'WN': 'Southwest Airlines', 'B6': 'JetBlue Airways', 'AS': 'Alaska Airlines',
    'NK': 'Spirit Airlines', 'F9': 'Frontier Airlines', 'G4': 'Allegiant Air',
    'HA': 'Hawaiian Airlines', 'SY': 'Sun Country Airlines', 'MX': 'Breeze Airways',
    'MQ': 'American Eagle', 'OO': 'SkyWest Airlines', 'YX': 'Republic Airways',
    'QX': 'Horizon Air', '9E': 'Endeavor Air', 'OH': 'PSA Airlines',
    'YV': 'Mesa Airlines', 'PT': 'Piedmont Airlines', 'ZW': 'Air Wisconsin',
    'G7': 'GoJet Airlines', 'C5': 'CommutAir',
}

QUARTER_LABELS = {'Q1 (Jan-Mar)': 1, 'Q2 (Apr-Jun)': 2,
                  'Q3 (Jul-Sep)': 3, 'Q4 (Oct-Dec)': 4}

def carrier_tier_fn(code):
    if code in art['legacy_set']: return 0
    if code in art['lcc_set']:    return 1
    if code in art['ulcc_set']:   return 2
    return 3

def lookup_both_directions(d, c1, c2):
    return d.get((c1, c2)) or d.get((c2, c1))

def estimate_fare(city1, city2, carrier_label, year, quarter_label):
    if city1 == city2:
        return "Origin and destination must be different cities."

    carrier_code = carrier_label.split('(')[-1].replace(')', '').strip()
    quarter = QUARTER_LABELS[quarter_label]

    # Route-level lookups
    distance   = lookup_both_directions(art['distance_lookup'],   city1, city2)
    passengers = lookup_both_directions(art['passengers_lookup'], city1, city2)
    large_ms   = lookup_both_directions(art['ms_lookup'],         city1, city2)

    route_known = distance is not None
    if not route_known:
        distance, passengers, large_ms = 800, 500, 0.5

    distance   = int(round(distance))
    passengers = int(round(passengers))

    try:
        carrier_enc = art['carrier_encoder'].transform([carrier_code])[0]
    except ValueError:
        return f"Carrier {carrier_code} not in training data."

    # Macro-level lookup (fuel + CPI for this year/quarter)
    macro = art['macro_lookup'].get((int(year), int(quarter)),
                                      art['macro_defaults'])

    row = pd.DataFrame([{
        # DOT passthrough
        'Year': year, 'quarter': quarter,
        'nsmiles': distance, 'passengers': passengers, 'large_ms': large_ms,
        # DOT-derived engineered
        'log_distance':    np.log1p(distance),
        'log_passengers':  np.log1p(passengers),
        'is_long_haul':    int(distance > 1500),
        'distance_bucket': int(pd.cut([distance], bins=[0, 500, 1000, 1500, np.inf],
                                      labels=[0, 1, 2, 3])[0]),
        'quarter_sin':     np.sin(2 * np.pi * quarter / 4),
        'quarter_cos':     np.cos(2 * np.pi * quarter / 4),
        'is_covid_era':    int(year in (2020, 2021)),
        'carrier_tier':    carrier_tier_fn(carrier_code),
        'carrier_lg_enc':  carrier_enc,
        'city1_freq':      art['city1_freq'].get(city1, 0.001),
        'city2_freq':      art['city2_freq'].get(city2, 0.001),
        'route_popularity': art['route_pop_map'].get((city1, city2),
                             art['route_pop_map'].get((city2, city1),
                             art['global_mean_pop'])),
        # External passthrough (fuel + CPI)
        'jet_fuel_usd_per_gal': macro['jet_fuel_usd_per_gal'],
        'jet_fuel_lag_1q':      macro['jet_fuel_lag_1q'],
        'jet_fuel_yoy_change':  macro['jet_fuel_yoy_change'],
        'cpi':                  macro['cpi'],
        'cpi_yoy_change':       macro['cpi_yoy_change'],
    }])[art['feature_cols']]

    pred = art['model'].predict(row)[0]

    unknown_note = ("\n\nNote: Route not in DOT Top-1000 dataset. Defaults used; "
                    "estimate is less reliable.") if not route_known else ""

    return (f"Estimated Average Fare: ${pred:,.2f}\n"
            f"{'=' * 40}\n\n"
            f"Route:    {city1} -> {city2}\n"
            f"Distance: {distance:,} miles\n"
            f"Period:   {quarter_label} {year}\n"
            f"Airline:  {CARRIER_NAMES.get(carrier_code, carrier_code)}\n"
            f"Avg quarterly passengers on route: {passengers:,}\n"
            f"Largest-carrier market share: {large_ms*100:.0f}%\n\n"
            f"Macro context: jet fuel ${macro['jet_fuel_usd_per_gal']:.2f}/gal, "
            f"CPI {macro['cpi']:.1f}"
            f"{unknown_note}\n\n"
            f"This is a historical-pattern estimate, NOT a forecast. "
            f"Year input is capped at the training range (2010-2025). "
            f"See notebook section 10.5 for why this model cannot forecast.")

carrier_choices = [f"{CARRIER_NAMES.get(c, c)} ({c})" for c in art['carriers']]

demo = gr.Interface(
    fn=estimate_fare,
    inputs=[
        gr.Dropdown(art['cities'], label="Origin City", value="Chicago, IL"),
        gr.Dropdown(art['cities'], label="Destination City",
                    value="New York City, NY (Metropolitan Area)"),
        gr.Dropdown(carrier_choices, label="Airline",
                    value="United Airlines (UA)"),
        gr.Slider(2010, 2025, step=1, value=2024, label="Year (training range only)"),
        gr.Dropdown(list(QUARTER_LABELS.keys()),
                    label="Quarter", value="Q2 (Apr-Jun)"),
    ],
    outputs=gr.Textbox(label="Fare Estimate", lines=16),
    title="US Domestic Fare Estimator",
    description=("Estimates historical average quarterly airfare for US domestic routes "
                 "using a tuned HistGradientBoosting model trained on DOT Consumer Airfare "
                 "Report data (2010-2025), augmented with EIA jet fuel and BLS CPI data. "
                 "This is estimation, not forecasting - see notebook section 10.5 for the "
                 "temporal-holdout analysis."),
)

demo.launch()


## 14. Summary & Rubric Mapping

| Rubric requirement | Notebook section |
|---|---|
| >= 1,000 samples (CS 451) | §1 — ~40K rows post-filtering |
| Non-toy dataset with quality issues | §1.2 — outliers, NaN, leakage audited |
| >= 5 EDA visualizations with interpretation | §3 — 7 visualizations (+ 2 more in §2.5.6) |
| Handle missing / duplicates / outliers | §2 — explicit + capped |
| Before/after data quality comparison | §2 — quantified table |
| **Additional data sources merged** (originality) | **§2.5 — EIA jet fuel + BLS CPI integrated** |
| >= 3 engineered features | §4 — 13 features (8 DOT-derived + 5 external) |
| Feature selection technique | §9 — permutation importance |
| >= 2 different model families | §7 — Ridge + RF + HistGB (3 families) |
| Proper train/val/test split | §6 — 60/20/20, leakage-safe |
| Systematic hyperparameter tuning | §8 — RandomizedSearchCV |
| Multiple evaluation metrics | §10 — R2, MAE, RMSE, MAPE |
| CV variance estimates | §7 — 5-fold CV mean +/- std |
| Model comparison table + chart | §11 |
| **Ablation study** (grad-level bonus) | **§10.6 — DOT-only vs DOT+external** |
| **Honest failure-case analysis** | **§10.5 — temporal holdout shows forecasting failure** |
| Working deployment | §13 — Gradio estimator app |

### Key contributions beyond a standard pipeline

Three contributions differentiate this project from a standard regression exercise:

1. **External data integration (§2.5):** We merged two public macroeconomic sources (EIA jet fuel, BLS CPI) with the primary DOT dataset, giving the model visibility into costs and inflation it would otherwise lack.

2. **Ablation study (§10.6):** We quantify how much the external data actually helps by training otherwise-identical models with and without the new features and reporting both on the held-out test set.

3. **Honest failure-case analysis (§10.5):** Most Kaggle-style notebooks stop at the random-split test score and claim the model "predicts" fares. We show empirically that the same model's performance degrades substantially when asked to forecast — and we honestly reframe the deployment around what the model can actually do (estimate historical patterns) rather than what it can't (forecast the future).

**Reproducibility:** All random seeds fixed to `RANDOM_STATE = 42`. Data pulled live from DOT Socrata API, EIA API, and BLS API. No local files required.